<a href="https://colab.research.google.com/github/Andrian0s/ML4NLP1-2026-Tutorial-Notebooks/blob/main/exercises/ex2/ex2_word_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML4NLP1
## Starting Point for Exercise 2

This notebook is supposed to serve as a starting point and/or inspiration when starting exercise 2. It begins with the PyTorch tutorial on word embeddings; the exercise itself starts at *Exercise: Computing Word Embeddings: Continuous Bag-of-Words*.

Throughout the notebook, there are questions that you should address in your report. These are marked with 📝❓ and are collected again in the lab report section at the end of the notebook.

## Use of generative AI

AI tools may be used in this exercise to help you explain code, debug, improve documentation, and explore alternatives. Any use must be documented in the *Declaration on the use of AI* at the end of this notebook (which tools were used and for which purposes). You remain responsible for understanding, checking, and testing everything you submit, and you must be able to explain any submitted text, code, analyses, and AI-assisted contributions. See the [CL guidelines on academic integrity and AI use](https://www.cl.uzh.ch/en/Studies/Services/academic-integrity.html).

# Setup

In [1]:
%pip install -q gensim==4.4.0 gdown

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='Glyph .* missing from font')
warnings.filterwarnings('ignore', message='.*IProgress not found.*')
import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

import torch
# Pick the best available device: NVIDIA GPU, then Apple Silicon GPU, then CPU
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(device)

Note: you may need to restart the kernel to use updated packages.
cuda


In [3]:
%matplotlib inline

In [ ]:
# run this cell only if you're working with Google Colab
from google.colab import drive
drive.mount('/content/drive')

### Source (adapted): [link](https://pytorch.org/tutorials/beginner/nlp/word_embeddings_tutorial.html#exercise-computing-word-embeddings-continuous-bag-of-words)

# Word Embeddings: Encoding Lexical Semantics

Word embeddings are dense vectors of real numbers, one per word in your
vocabulary. In NLP, it is almost always the case that your features are
words! But how should you represent a word in a computer? You could
store its ascii character representation, but that only tells you what
the word *is*, it doesn't say much about what it *means* (you might be
able to derive its part of speech from its affixes, or properties from
its capitalization, but not much). Even more, in what sense could you
combine these representations? We often want dense outputs from our
neural networks, where the inputs are $|V|$ dimensional, where
$V$ is our vocabulary, but often the outputs are only a few
dimensional (if we are only predicting a handful of labels, for
instance). How do we get from a massive dimensional space to a smaller
dimensional space?

How about instead of ascii representations, we use a one-hot encoding?
That is, we represent the word $w$ by

\begin{align}\overbrace{\left[ 0, 0, \dots, 1, \dots, 0, 0 \right]}^\text{|V| elements}\end{align}

where the 1 is in a location unique to $w$. Any other word will
have a 1 in some other location, and a 0 everywhere else.

There is an enormous drawback to this representation, besides just how
huge it is. It basically treats all words as independent entities with
no relation to each other. What we really want is some notion of
*similarity* between words. Why? Let's see an example.

Suppose we are building a language model. Suppose we have seen the
sentences

* The mathematician ran to the store.
* The physicist ran to the store.
* The mathematician solved the open problem.

in our training data. Now suppose we get a new sentence never before
seen in our training data:

* The physicist solved the open problem.

Our language model might do OK on this sentence, but wouldn't it be much
better if we could use the following two facts:

* We have seen  mathematician and physicist in the same role in a sentence. Somehow they
  have a semantic relation.
* We have seen mathematician in the same role  in this new unseen sentence
  as we are now seeing physicist.

and then infer that physicist is actually a good fit in the new unseen
sentence? This is what we mean by a notion of similarity: we mean
*semantic similarity*, not simply having similar orthographic
representations. It is a technique to combat the sparsity of linguistic
data, by connecting the dots between what we have seen and what we
haven't. This example of course relies on a fundamental linguistic
assumption: that words appearing in similar contexts are related to each
other semantically. This is called the [distributional
hypothesis](https://en.wikipedia.org/wiki/Distributional_semantics).


# Getting Dense Word Embeddings

How can we solve this problem? That is, how could we actually encode
semantic similarity in words? Maybe we think up some semantic
attributes. For example, we see that both mathematicians and physicists
can run, so maybe we give these words a high score for the "is able to
run" semantic attribute. Think of some other attributes, and imagine
what you might score some common words on those attributes.

If each attribute is a dimension, then we might give each word a vector,
like this:

\begin{align}q_\text{mathematician} = \left[ \overbrace{2.3}^\text{can run},
   \overbrace{9.4}^\text{likes coffee}, \overbrace{-5.5}^\text{majored in Physics}, \dots \right]\end{align}

\begin{align}q_\text{physicist} = \left[ \overbrace{2.5}^\text{can run},
   \overbrace{9.1}^\text{likes coffee}, \overbrace{6.4}^\text{majored in Physics}, \dots \right]\end{align}

Then we can get a measure of similarity between these words by doing:

\begin{align}\text{Similarity}(\text{physicist}, \text{mathematician}) = q_\text{physicist} \cdot q_\text{mathematician}\end{align}

Although it is more common to normalize by the lengths:

\begin{align}\text{Similarity}(\text{physicist}, \text{mathematician}) = \frac{q_\text{physicist} \cdot q_\text{mathematician}}
   {\| q_\text{physicist} \| \| q_\text{mathematician} \|} = \cos (\phi)\end{align}

Where $\phi$ is the angle between the two vectors. That way,
extremely similar words (words whose embeddings point in the same
direction) will have similarity 1. Extremely dissimilar words should
have similarity -1.


You can think of the sparse one-hot vectors from the beginning of this
section as a special case of these new vectors we have defined, where
each word basically has similarity 0, and we gave each word some unique
semantic attribute. These new vectors are *dense*, which is to say their
entries are (typically) non-zero.

But these new vectors are a big pain: you could think of thousands of
different semantic attributes that might be relevant to determining
similarity, and how on earth would you set the values of the different
attributes? Central to the idea of deep learning is that the neural
network learns representations of the features, rather than requiring
the programmer to design them herself. So why not just let the word
embeddings be parameters in our model, and then be updated during
training? This is exactly what we will do. We will have some *latent
semantic attributes* that the network can, in principle, learn. Note
that the word embeddings will probably not be interpretable. That is,
although with our hand-crafted vectors above we can see that
mathematicians and physicists are similar in that they both like coffee,
if we allow a neural network to learn the embeddings and see that both
mathematicians and physicists have a large value in the second
dimension, it is not clear what that means. They are similar in some
latent semantic dimension, but this probably has no interpretation to
us.


In summary, **word embeddings are a representation of the *semantics* of
a word, efficiently encoding semantic information that might be relevant
to the task at hand**. You can embed other things too: part of speech
tags, parse trees, anything! The idea of feature embeddings is central
to the field.


# Word Embeddings in Pytorch

Before we get to a worked example and an exercise, a few quick notes
about how to use embeddings in Pytorch and in deep learning programming
in general. Similar to how we defined a unique index for each word when
making one-hot vectors, we also need to define an index for each word
when using embeddings. These will be keys into a lookup table. That is,
embeddings are stored as a $|V| \times D$ matrix, where $D$
is the dimensionality of the embeddings, such that the word assigned
index $i$ has its embedding stored in the $i$'th row of the
matrix. In all of my code, the mapping from words to indices is a
dictionary named word\_to\_ix.

The module that allows you to use embeddings is torch.nn.Embedding,
which takes two arguments: the vocabulary size, and the dimensionality
of the embeddings.

To index into this table, you must use a tensor of integer indices,
usually created with `torch.tensor(..., dtype=torch.long)` (the indices are integers, not floats).




In [4]:
# Author: Robert Guthrie

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
torch.manual_seed(1)

In [5]:
word_to_ix = {"hello": 0, "world": 1}
embeds = nn.Embedding(2, 5)  # 2 words in vocab, 5 dimensional embeddings
lookup_tensor = torch.tensor([word_to_ix["hello"]], dtype=torch.long)
hello_embed = embeds(lookup_tensor)
print(hello_embed)

tensor([[ 0.6614,  0.2669,  0.0617,  0.6213, -0.4519]],
       grad_fn=<EmbeddingBackward0>)


# An Example: N-Gram Language Modeling

Recall that in an n-gram language model, given a sequence of words
$w$, we want to compute

\begin{align}P(w_i | w_{i-1}, w_{i-2}, \dots, w_{i-n+1} )\end{align}

Where $w_i$ is the ith word of the sequence.

In this example, we will compute the loss function on some training
examples and update the parameters with backpropagation.

In [6]:
CONTEXT_SIZE = 2
EMBEDDING_DIM = 10
# We will use Shakespeare Sonnet 2
test_sentence = """When forty winters shall besiege thy brow,
And dig deep trenches in thy beauty's field,
Thy youth's proud livery so gazed on now,
Will be a totter'd weed of small worth held:
Then being asked, where all thy beauty lies,
Where all the treasure of thy lusty days;
To say, within thine own deep sunken eyes,
Were an all-eating shame, and thriftless praise.
How much more praise deserv'd thy beauty's use,
If thou couldst answer 'This fair child of mine
Shall sum my count, and make my old excuse,'
Proving his beauty by succession thine!
This were to be new made when thou art old,
And see thy blood warm when thou feel'st it cold.""".split()
# we should tokenize the input, but we will ignore that for now
# build a list of tuples.  Each tuple is ([ word_i-2, word_i-1 ], target word)
trigrams = [([test_sentence[i], test_sentence[i + 1]], test_sentence[i + 2])
            for i in range(len(test_sentence) - 2)]
# print the first 3, just so you can see what they look like
print(trigrams[:3])

vocab = sorted(set(test_sentence))  # sorted, so that every word gets the same index in every session
word_to_ix = {word: i for i, word in enumerate(vocab)}


class NGramLanguageModeler(nn.Module):

    def __init__(self, vocab_size, embedding_dim, context_size):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(context_size * embedding_dim, 128)
        self.linear2 = nn.Linear(128, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs).view((1, -1))
        out = F.relu(self.linear1(embeds))
        logits = self.linear2(out)  # raw scores over the vocabulary; the softmax is applied inside the loss
        return logits


losses = []
loss_function = nn.CrossEntropyLoss()  # takes logits and class indices
model = NGramLanguageModeler(len(vocab), EMBEDDING_DIM, CONTEXT_SIZE)
optimizer = optim.SGD(model.parameters(), lr=0.001)

for epoch in range(10):
    total_loss = 0
    for context, target in trigrams:

        # Step 1. Prepare the inputs to be passed to the model (i.e, turn the words
        # into integer indices and wrap them in tensors)
        context_idxs = torch.tensor([word_to_ix[w] for w in context], dtype=torch.long)

        # Step 2. Recall that torch *accumulates* gradients. Before passing in a
        # new instance, you need to zero out the gradients from the old
        # instance
        model.zero_grad()

        # Step 3. Run the forward pass, getting logits (unnormalised scores) over next
        # words
        logits = model(context_idxs)

        # Step 4. Compute your loss function. (Again, Torch wants the target
        # word wrapped in a tensor)
        loss = loss_function(logits, torch.tensor([word_to_ix[target]], dtype=torch.long))

        # Step 5. Do the backward pass and update the gradient
        loss.backward()
        optimizer.step()

        # Get the Python number from a 1-element Tensor by calling tensor.item()
        total_loss += loss.item()
    mean_loss = total_loss / len(trigrams)  # average loss per training example
    print("Loss in Epoch {ep}: {l}".format(ep=epoch, l=np.round(mean_loss, 3))) # The loss decreased every iteration over the training data!
    losses.append(mean_loss)

[(['When', 'forty'], 'winters'), (['forty', 'winters'], 'shall'), (['winters', 'shall'], 'besiege')]
Loss in Epoch 0: 4.599
Loss in Epoch 1: 4.576
Loss in Epoch 2: 4.553
Loss in Epoch 3: 4.531
Loss in Epoch 4: 4.509
Loss in Epoch 5: 4.486
Loss in Epoch 6: 4.464
Loss in Epoch 7: 4.442
Loss in Epoch 8: 4.42
Loss in Epoch 9: 4.399


# Exercise: Computing Word Embeddings: Continuous Bag-of-Words

The Continuous Bag-of-Words model (CBOW) is frequently used in NLP deep
learning. It is a model that tries to predict words given the context of
a few words before and a few words after the target word. This is
distinct from language modeling, since CBOW is not sequential and does
not have to be probabilistic. Typically, CBOW is used to quickly train
word embeddings, and these embeddings are used to initialize the
embeddings of some more complicated model. Usually, this is referred to
as *pretraining embeddings*. It almost always helps performance a couple
of percent.

The CBOW model is as follows. Given a target word $w_i$ and an
$N$ context window on each side, $w_{i-1}, \dots, w_{i-N}$
and $w_{i+1}, \dots, w_{i+N}$, referring to all context words
collectively as $C$, CBOW tries to minimize

\begin{align}-\log p(w_i | C) = -\log \text{Softmax}(A(\sum_{w \in C} q_w) + b)\end{align}

where $q_w$ is the embedding for word $w$.


## Exercise Layout
### 1. <u>Training CBOW Embeddings</u>
1.1) Implement a CBOW Model by completing ```class CBOW(nn.Module)``` and test it on ```raw_text```.    

1.2) Load Datasets ```tripadvisor_hotel_reviews_reduced.csv``` and ```scifi_reduced.txt```.     

1.3) Decide preprocessing steps by completing the function ```def custom_preprocess()```. Describe your decisions. Note that it's your choice to create different preprocessing functions for hotel reviews and scifi datasets or use the same preprocessing function.             

1.4) Train CBOW2 with a context width of 2 (in both directions) for the Hotel Reviews dataset.   

1.5) Train CBOW5 with a context width of 5 (in both directions) for the Hotel Reviews dataset. Are predictions made by the model sensitive towards the context size?
     
1.6) Train CBOW2 with a context width of 2 (in both directions) for the Sci-Fi story dataset.  


### 2. <u>Test your Embeddings</u>
Note - Do the following for CBOW2, and optionally for CBOW5

2.1) For the hotel reviews dataset, choose 3 nouns, 3 verbs, and 3 adjectives. Make sure that some nouns/verbs/adjectives occur frequently in the corpus and that others are rare. For each of the 9 chosen words, retrieve the 5 closest words according to your trained CBOW2 model. List them in your report and comment on the performance of your model: do the neighbours the model provides make sense? Discuss.   

2.2) Do the same for Sci-Fi dataset.   

2.3) How does the quality of the hotel review-based embeddings compare with the Sci-fi-based embeddings? Elaborate.   

2.4) Choose 2 words and retrieve their 5 closest neighbours according to hotel review-based embeddings and the Sci-fi-based embeddings. Do they have different neighbours? If yes, can you reason why?    

2.5) What are the differences between CBOW2 and CBOW5 ? Can you "describe" them?   

### 3. <u>Compare with pretrained embeddings</u>
3.1) Load the pretrained GloVe embeddings with the given code snippet and retrieve the 5 closest neighbours for your selection of words. Compare them with the neighbours from your CBOW2 (and optionally CBOW5) models: Do the GloVe neighbours seem more general or semantically broader? Are the CBOW neighbours more domain-specific to hotel reviews or the Sci-Fi text? Discuss how training data size and training objective (predictive vs. count-based) might explain the observed differences.


### Tips

1. Switch from CPU to a GPU instance after you have confirmed that your training procedure is working correctly.
2. You can always save your intermediate results (embeddings, preprocessed dataset, model, etc.) in your google drive via colab
3. For the hotel reviews and the Sci-Fi stories, use a batch size of 128–512: training then takes minutes instead of hours. With larger batches, you may need a larger learning rate.



### 1.1 Create a CBOW Model by completing ```class CBOW(nn.Module)``` and test it on ```raw_text```
Implement CBOW in Pytorch by filling in the class below. Some
tips:

* Think about which parameters you need to define.
* Make sure you know what shape each operation expects. Use .view() if you need to
  reshape.

In [7]:
CONTEXT_SIZE = 2  # 2 words to the left, 2 to the right
raw_text = """We are about to study the idea of a computational process.
Computational processes are abstract beings that inhabit computers.
As they evolve, processes manipulate other abstract things called data.
The evolution of a process is directed by a pattern of rules
called a program. People create programs to direct processes. In effect,
we conjure the spirits of the computer with our spells.""".split()

# By deriving a set from `raw_text`, we deduplicate the array;
# sorting it gives every word the same index in every session
vocab = sorted(set(raw_text))
vocab_size = len(vocab)

word_to_ix = {word: i for i, word in enumerate(vocab)}
data = []
for i in range(2, len(raw_text) - 2):
    context = [raw_text[i - 2], raw_text[i - 1],
               raw_text[i + 1], raw_text[i + 2]]
    target = raw_text[i]
    data.append((context, target))
print(data[:5])


class CBOW(nn.Module):

    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, inputs):
        if inputs.dim() == 1:
            inputs = inputs.unsqueeze(0)
        embeds = self.embeddings(inputs).sum(dim=1)
        logits = self.linear(embeds)
        return logits


def make_context_vector(context, word_to_ix):
    idxs = [word_to_ix[w] for w in context]
    return torch.tensor(idxs, dtype=torch.long)


make_context_vector(data[0][0], word_to_ix)  # example

[(['We', 'are', 'to', 'study'], 'about'), (['are', 'about', 'study', 'the'], 'to'), (['about', 'to', 'the', 'idea'], 'study'), (['to', 'study', 'idea', 'of'], 'the'), (['study', 'the', 'of', 'a'], 'idea')]


tensor([ 5,  9, 46, 41])

In [8]:
### here are some functions to help you make the data ready for use by your model
from torch.utils.data import TensorDataset, DataLoader

# Function to turn the (context, target) pairs into index tensors
# Input: list of tuples (list of context words, target word)
# Output: contexts of shape (N, 2 * CONTEXT_SIZE) and targets of shape (N,), both torch.long
def make_tensors(data, word_to_ix):
    contexts = torch.tensor([[word_to_ix[w] for w in context] for context, _ in data], dtype=torch.long)
    targets = torch.tensor([word_to_ix[target] for _, target in data], dtype=torch.long)
    return contexts, targets

# Function to create a DataLoader that serves batches of (contexts, targets)
# Use TensorDataset and DataLoader, as in the W05 tutorial
def make_loader(data, word_to_ix, batch_size=128, shuffle=True):
    contexts, targets = make_tensors(data, word_to_ix)
    dataset = TensorDataset(contexts, targets)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

In [9]:
# Function to train the model
def train(model, loader, device, NUM_EPOCHS=15, lr=0.003):
    model = model.to(device)
    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    losses = []

    for epoch in range(1, NUM_EPOCHS + 1):
        total_loss = 0.0
        for contexts, targets in loader:
            contexts, targets = contexts.to(device), targets.to(device)
            optimizer.zero_grad()
            logits = model(contexts)
            loss = loss_function(logits, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * contexts.size(0)

        mean_loss = total_loss / len(loader.dataset)
        losses.append(mean_loss)
        print(f"Epoch {epoch:2d}/{NUM_EPOCHS} - Loss: {mean_loss:.4f}")

    return losses

In [10]:
### create your model and train
EMBEDDING_DIM = 50
toy_loader = make_loader(data, word_to_ix, batch_size=16, shuffle=True)
toy_model = CBOW(vocab_size, EMBEDDING_DIM)
toy_losses = train(toy_model, toy_loader, device, NUM_EPOCHS=15, lr=0.01)

Epoch  1/15 - Loss: 4.3902
Epoch  2/15 - Loss: 2.6245
Epoch  3/15 - Loss: 1.5391
Epoch  4/15 - Loss: 0.8921
Epoch  5/15 - Loss: 0.5026
Epoch  6/15 - Loss: 0.2874
Epoch  7/15 - Loss: 0.1758
Epoch  8/15 - Loss: 0.1159
Epoch  9/15 - Loss: 0.0810
Epoch 10/15 - Loss: 0.0607
Epoch 11/15 - Loss: 0.0476
Epoch 12/15 - Loss: 0.0381
Epoch 13/15 - Loss: 0.0322
Epoch 14/15 - Loss: 0.0281
Epoch 15/15 - Loss: 0.0251


### 1.2 Load Datasets

In [11]:
import pandas as pd

In [12]:
# Download Datasets tripadvisor_hotel_reviews_reduced.csv and scifi_reduced.txt
import os

hotel_url = "https://drive.google.com/uc?id=1foE1JuZJeu5E_4qVge9kExzhvF32teuF"
hotel_file = "content/tripadvisor_hotel_reviews_reduced.csv"
scifi_url = "https://drive.google.com/uc?id=13IWXrTjGTrfCd9l7dScZVO8ZvMicPU75"
scifi_file = "content/scifi_reduced.txt"

# Check if datasets are already present
if os.path.exists(hotel_file) and os.path.exists(scifi_file):
    print("Datasets already exist in content/ directory.")
else:
    try:
        # First attempt: original shell commands (Colab / Linux)
        !mkdir -p content
        !gdown "https://drive.google.com/uc?id=1foE1JuZJeu5E_4qVge9kExzhvF32teuF" -O "content/tripadvisor_hotel_reviews_reduced.csv"
        !gdown "https://drive.google.com/uc?id=13IWXrTjGTrfCd9l7dScZVO8ZvMicPU75" -O "content/scifi_reduced.txt"

        if not (os.path.exists(hotel_file) and os.path.exists(scifi_file)):
            raise RuntimeError("Shell download did not produce the expected files.")
        print("Successfully downloaded datasets via shell command.")
    except Exception as e:
        # Fallback for local Windows machines: use python-native gdown
        print(f"Shell command failed ({e}). Using Python gdown workaround for local Windows machine...")
        import gdown
        os.makedirs("content", exist_ok=True)
        if not os.path.exists(hotel_file):
            gdown.download(hotel_url, hotel_file, quiet=False)
        if not os.path.exists(scifi_file):
            gdown.download(scifi_url, scifi_file, quiet=False)
        print("Successfully downloaded datasets via Python gdown fallback.")


Datasets already exist in content/ directory.


In [13]:
### TODO: Load the datasets
hotel_df = pd.read_csv(hotel_file)
hotel_reviews = hotel_df['Review'].dropna().tolist()
print(f"Loaded {len(hotel_reviews):,} hotel reviews.")

with open(scifi_file, "r", encoding="utf-8") as f:
    scifi_raw_text = f.read()
print(f"Loaded Sci-Fi corpus ({len(scifi_raw_text):,} characters).")

Loaded 10,000 hotel reviews.
Loaded Sci-Fi corpus (43,062,636 characters).


### 1.3 Preprocess Datasets
📝❓ Describe your decisions for preprocessing the datasets and for the structure of your classes.

In [14]:
### Import libraries for preprocessing
import re
import json
from collections import Counter

In [15]:
### Complete the preprocessing function and apply it to the datasets
def custom_preprocess(data):
    if isinstance(data, str):
        sentences = re.split(r'[.!?]+', data)
    else:
        sentences = data

    tokenized_sentences = []
    for sentence in sentences:
        tokens = re.findall(r'[a-z]+', sentence.lower())
        if len(tokens) >= 5:
            tokenized_sentences.append(tokens)
    return tokenized_sentences

hotel_tokenized = custom_preprocess(hotel_reviews)
scifi_tokenized = custom_preprocess(scifi_raw_text)

print(f"Hotel preprocessed: {len(hotel_tokenized):,} reviews, example: {hotel_tokenized[0][:10]}")
print(f"Sci-Fi preprocessed: {len(scifi_tokenized):,} sentences, example: {scifi_tokenized[0][:10]}")

Hotel preprocessed: 10,000 reviews, example: ['fantastic', 'service', 'large', 'hotel', 'caters', 'business', 'corporates', 'serve', 'provided', 'better']
Sci-Fi preprocessed: 521,896 sentences, example: ['a', 'chat', 'with', 'the', 'editor', 'i', 'science', 'fiction', 'magazine', 'called']


In [16]:
### Function to get vocab
# Input will be a list of lists
# Output - vocab set
def get_vocab(raw_llist):
    return sorted(set(word for sentence in raw_llist for word in sentence))

### Function to get word-to-ix dictionary
def get_word2ix(vocab, save_loc=None):
    word_to_ix = {word: i for i, word in enumerate(sorted(vocab))}
    if save_loc is not None:
        with open(save_loc, "w", encoding="utf-8") as f:
            json.dump(word_to_ix, f)
    return word_to_ix

In [17]:
### Function to generate tuples of context-target
# Input = list of lists
# Output = list of tuples (list of context_words, target)
def get_data(raw_llist, context_window=5):
    data = []
    for sentence in raw_llist:
        for i in range(context_window, len(sentence) - context_window):
            context = sentence[i - context_window : i] + sentence[i + 1 : i + context_window + 1]
            target = sentence[i]
            data.append((context, target))
    return data

### 1.4 Train CBOW2 with a context width of 2 (in both directions) for the Hotel Reviews dataset.

In [18]:
EMBEDDING_DIM = 50
BATCH_SIZE = 512

hotel_vocab = get_vocab(hotel_tokenized)
hotel_word2ix = get_word2ix(hotel_vocab, save_loc="content/hotel_word2ix.json")
hotel_ix2word = {i: w for w, i in hotel_word2ix.items()}

hotel_data_cbow2 = get_data(hotel_tokenized, context_window=2)
print(f"Hotel vocab size: {len(hotel_vocab):,}, CBOW2 training pairs: {len(hotel_data_cbow2):,}")

hotel_loader_cbow2 = make_loader(hotel_data_cbow2, hotel_word2ix, batch_size=BATCH_SIZE)
model_hotel_cbow2 = CBOW(len(hotel_vocab), EMBEDDING_DIM)
losses_hotel_cbow2 = train(model_hotel_cbow2, hotel_loader_cbow2, device, NUM_EPOCHS=15)

Hotel vocab size: 34,536, CBOW2 training pairs: 1,023,066
Epoch  1/15 - Loss: 7.4463
Epoch  2/15 - Loss: 6.6000
Epoch  3/15 - Loss: 6.2888
Epoch  4/15 - Loss: 6.0847
Epoch  5/15 - Loss: 5.9305
Epoch  6/15 - Loss: 5.8075
Epoch  7/15 - Loss: 5.7031
Epoch  8/15 - Loss: 5.6124
Epoch  9/15 - Loss: 5.5331
Epoch 10/15 - Loss: 5.4618
Epoch 11/15 - Loss: 5.3975
Epoch 12/15 - Loss: 5.3405
Epoch 13/15 - Loss: 5.2882
Epoch 14/15 - Loss: 5.2410
Epoch 15/15 - Loss: 5.1990


### 1.5 Train CBOW5 with a context width of 5 (in both directions) for the Hotel Reviews dataset.  

📝❓ Are predictions made by the model sensitive towards the context size?

In [19]:
hotel_data_cbow5 = get_data(hotel_tokenized, context_window=5)
print(f"Hotel CBOW5 training pairs: {len(hotel_data_cbow5):,}")

hotel_loader_cbow5 = make_loader(hotel_data_cbow5, hotel_word2ix, batch_size=BATCH_SIZE)
model_hotel_cbow5 = CBOW(len(hotel_vocab), EMBEDDING_DIM)
losses_hotel_cbow5 = train(model_hotel_cbow5, hotel_loader_cbow5, device, NUM_EPOCHS=15)

Hotel CBOW5 training pairs: 963,073
Epoch  1/15 - Loss: 7.6886
Epoch  2/15 - Loss: 6.8598
Epoch  3/15 - Loss: 6.5681
Epoch  4/15 - Loss: 6.3677
Epoch  5/15 - Loss: 6.2104
Epoch  6/15 - Loss: 6.0741
Epoch  7/15 - Loss: 5.9529
Epoch  8/15 - Loss: 5.8433
Epoch  9/15 - Loss: 5.7431
Epoch 10/15 - Loss: 5.6501
Epoch 11/15 - Loss: 5.5663
Epoch 12/15 - Loss: 5.4876
Epoch 13/15 - Loss: 5.4164
Epoch 14/15 - Loss: 5.3508
Epoch 15/15 - Loss: 5.2907


### 1.6 Train CBOW2 with a context width of 2 (in both directions) for the Sci-Fi story dataset

In [27]:
scifi_vocab = get_vocab(scifi_tokenized)
scifi_word2ix = get_word2ix(scifi_vocab, save_loc="content/scifi_word2ix.json")
scifi_ix2word = {i: w for w, i in scifi_word2ix.items()}

scifi_data_cbow2 = get_data(scifi_tokenized, context_window=2)
print(f"Sci-Fi vocab size: {len(scifi_vocab):,}, CBOW2 training pairs: {len(scifi_data_cbow2):,}")

scifi_loader_cbow2 = make_loader(scifi_data_cbow2, scifi_word2ix, batch_size=BATCH_SIZE)
model_scifi_cbow2 = CBOW(len(scifi_vocab), EMBEDDING_DIM)
losses_scifi_cbow2 = train(model_scifi_cbow2, scifi_loader_cbow2, device, NUM_EPOCHS=3)

Sci-Fi vocab size: 108,185, CBOW2 training pairs: 5,373,485
Epoch  1/3 - Loss: 6.3908
Epoch  2/3 - Loss: 5.8605
Epoch  3/3 - Loss: 5.6855


### Function for nearest neighbour computation

Complete the following function and use it to retrieve the closest words in Part 2. It should rank all words in the vocabulary by the cosine similarity of their embeddings to the embedding of the query word, the same measure that GloVe's `most_similar()` uses in Part 3. You may adapt the arguments to your model and vocabulary.

In [28]:
def get_closest_words(word, embedding, word_to_ix, ix_to_word, topn=5):
    """Return the topn words whose embeddings are most similar (cosine similarity) to the embedding of `word`.

    embedding:  the nn.Embedding layer of your trained model (e.g. model.embeddings)
    word_to_ix: dictionary from words to indices
    ix_to_word: dictionary (or list) from indices to words
    Returns a list of (word, similarity) pairs, most similar first, without the query word itself.
    """
    if word not in word_to_ix:
        return []

    weights = embedding.weight.detach()
    query_idx = word_to_ix[word]
    query_vec = weights[query_idx].unsqueeze(0)

    similarities = F.cosine_similarity(query_vec, weights, dim=1)
    top_scores, top_indices = torch.topk(similarities, k=topn + 1)

    results = []
    for idx, score in zip(top_indices.tolist(), top_scores.tolist()):
        if idx != query_idx:
            results.append((ix_to_word[idx], score))
        if len(results) == topn:
            break
    return results

### 2.1 For the hotel reviews dataset, choose 3 nouns, 3 verbs, and 3 adjectives. (CBOW2 and optionally for CBOW5)
Make sure that some nouns/verbs/adjectives occur frequently in the corpus and that others are rare. For each of the 9 chosen words, retrieve the 5 closest words according to your trained CBOW2 model (you can use the function `get_closest_words` above).    

📝❓ List them in your report (at the end of this notebook) and comment on the performance of your model: do the neighbours the model provides make sense? Discuss.   


In [29]:
hotel_word_counts = Counter(w for s in hotel_tokenized for w in s)

# 3 nouns, 3 verbs, 3 adjectives (mix of frequent and rare words in the corpus)
hotel_words = {
    'Nouns': ['hotel', 'room', 'mattress'],         # frequent: hotel, room | rare: mattress
    'Verbs': ['stay', 'recommend', 'apologize'],    # frequent: stay, recommend | rare: apologize
    'Adjectives': ['great', 'clean', 'filthy'],     # frequent: great, clean | rare: filthy
}

for category, words in hotel_words.items():
    print(f"=== {category} (Hotel Reviews) ===")
    for w in words:
        cbow2_neighbors = get_closest_words(w, model_hotel_cbow2.embeddings, hotel_word2ix, hotel_ix2word, topn=5)
        cbow5_neighbors = get_closest_words(w, model_hotel_cbow5.embeddings, hotel_word2ix, hotel_ix2word, topn=5)
        print(f"\nWord: '{w}' (count = {hotel_word_counts[w]})")
        print("  CBOW2:", ", ".join(f"{nw} ({sim:.3f})" for nw, sim in cbow2_neighbors))
        print("  CBOW5:", ", ".join(f"{nw} ({sim:.3f})" for nw, sim in cbow5_neighbors))

=== Nouns (Hotel Reviews) ===

Word: 'hotel' (count = 24534)
  CBOW2: property (0.641), resort (0.609), dylan (0.606), ambilent (0.563), flophouse (0.549)
  CBOW5: property (0.606), contempated (0.591), place (0.561), commune (0.559), hotels (0.558)

Word: 'room' (count = 17122)
  CBOW2: rooms (0.784), patissserie (0.543), orleansthe (0.523), chiller (0.496), needahotel (0.495)
  CBOW5: rooms (0.817), spacous (0.541), vittoria (0.540), bemused (0.528), canals (0.522)

Word: 'mattress' (count = 112)
  CBOW2: beds (0.571), mattresses (0.544), bed (0.526), accomoations (0.523), pens (0.521)
  CBOW5: perhaps (0.547), prics (0.546), mattresses (0.529), greate (0.522), blankets (0.516)
=== Verbs (Hotel Reviews) ===

Word: 'stay' (count = 7563)
  CBOW2: sleep (0.616), return (0.578), staying (0.557), visit (0.555), spotlessley (0.521)
  CBOW5: staying (0.671), craibe (0.556), experience (0.546), fallings (0.531), vacation (0.524)

Word: 'recommend' (count = 2398)
  CBOW2: recommended (0.754),

### 2.2 Repeat 2.1 for SciFi Dataset

📝❓ List your findings for SciFi Dataset as well, similarly to 2.1

In [30]:
scifi_word_counts = Counter(w for s in scifi_tokenized for w in s)

# 3 nouns, 3 verbs, 3 adjectives (mix of frequent and rare words in the corpus)
scifi_words = {
    'Nouns': ['ship', 'planet', 'reactor'],            # frequent: ship, planet | rare: reactor
    'Verbs': ['said', 'looked', 'piloted'],            # frequent: said, looked | rare: piloted
    'Adjectives': ['human', 'alien', 'weightless'],    # frequent: human, alien | rare: weightless
}

for category, words in scifi_words.items():
    print(f"=== {category} (Sci-Fi Stories) ===")
    for w in words:
        cbow2_neighbors = get_closest_words(w, model_scifi_cbow2.embeddings, scifi_word2ix, scifi_ix2word, topn=5)
        print(f"\nWord: '{w}' (count = {scifi_word_counts[w]})")
        print("  CBOW2:", ", ".join(f"{nw} ({sim:.3f})" for nw, sim in cbow2_neighbors))

=== Nouns (Sci-Fi Stories) ===

Word: 'ship' (count = 6876)
  CBOW2: plane (0.797), car (0.796), vessel (0.730), craft (0.693), spaceship (0.672)

Word: 'planet' (count = 3584)
  CBOW2: world (0.758), grounds (0.703), island (0.673), journey (0.641), colony (0.638)

Word: 'reactor' (count = 54)
  CBOW2: flier (0.590), cuticula (0.587), minisub (0.578), firebox (0.570), marine (0.560)
=== Verbs (Sci-Fi Stories) ===

Word: 'said' (count = 30891)
  CBOW2: replied (0.821), whispered (0.818), answered (0.777), murmured (0.766), muttered (0.752)

Word: 'looked' (count = 7695)
  CBOW2: stared (0.869), glanced (0.860), glared (0.836), gazed (0.832), peered (0.751)

Word: 'piloted' (count = 24)
  CBOW2: macroscopic (0.609), scoutships (0.564), weirdt (0.548), unremembered (0.544), treesmell (0.537)
=== Adjectives (Sci-Fi Stories) ===

Word: 'human' (count = 3834)
  CBOW2: waster (0.626), intelligent (0.590), humanoid (0.586), dominant (0.576), raster (0.557)

Word: 'alien' (count = 1557)
  CBOW

### 2.3 📝❓ How does the quality of the hotel review-based embeddings compare with the Sci-fi-based embeddings? Elaborate.

### 2.4 Choose 2 words and retrieve their 5 closest neighbours according to hotel review-based embeddings and the Sci-fi-based embeddings.

📝❓ Do they have different neighbours? If yes, can you reason why?

In [31]:
shared_words = ['star', 'space']

for w in shared_words:
    hotel_nb = get_closest_words(w, model_hotel_cbow2.embeddings, hotel_word2ix, hotel_ix2word, topn=5)
    scifi_nb = get_closest_words(w, model_scifi_cbow2.embeddings, scifi_word2ix, scifi_ix2word, topn=5)
    print(f"Word: '{w}' (hotel count = {hotel_word_counts[w]}, scifi count = {scifi_word_counts[w]})")
    print("  Hotel CBOW2: ", ", ".join(f"{nw} ({sim:.3f})" for nw, sim in hotel_nb))
    print("  Sci-Fi CBOW2:", ", ".join(f"{nw} ({sim:.3f})" for nw, sim in scifi_nb))
    print()

Word: 'star' (hotel count = 1343, scifi count = 1168)
  Hotel CBOW2:  nadda (0.631), stars (0.618), comparing (0.555), alooha (0.541), googling (0.527)
  Sci-Fi CBOW2: trilogy (0.631), precipitating (0.621), wrol (0.588), freighter (0.583), beauties (0.574)

Word: 'space' (hotel count = 597, scifi count = 4725)
  Hotel CBOW2:  intramuscular (0.520), orsisni (0.515), eateries (0.511), instrumentalists (0.503), airconditioning (0.497)
  Sci-Fi CBOW2: prospero (0.601), bowmen (0.581), aboul (0.568), interplanetary (0.559), meanest (0.559)



### 2.5 📝❓ What are the differences between CBOW2 and CBOW5 ? Can you "describe" them?    

### 3. Use the following code snippet to load a pretrained GloVe embedding model
GloVe (Global Vectors for Word Representation) (link to paper: https://aclanthology.org/D14-1162/) is a count-based model trained on very large text corpora. The vectors used here (glove.6B, 50 dimensions, the same size we recommend for your CBOW embeddings) were trained on Wikipedia and the Gigaword newswire corpus, about 6 billion tokens in total. Unlike CBOW, which learns to predict a target word from its local context, GloVe learns embeddings that capture global co-occurrence statistics of words across the entire corpus.

Each word in GloVe has one static vector, i.e. its embedding does not change depending on context.

Note: Only training needs a GPU. Once your models are trained and saved, you can do Parts 2 and 3 in a CPU runtime.

In [32]:
import numpy as np

In [ ]:
import contextlib, io, os, sys, shutil
import gensim.downloader as api

# Clean up corrupted/incomplete gensim cache
glove_dir = os.path.join(api.BASE_DIR, "glove-wiki-gigaword-50")
if os.path.exists(glove_dir) and not os.path.exists(os.path.join(glove_dir, "__init__.py")):
    shutil.rmtree(glove_dir)
sys.modules.pop("glove-wiki-gigaword-50", None)

print("Loading GloVe (first run downloads ~66 MB, under a minute)...")
with contextlib.redirect_stdout(io.StringIO()):
    glove = api.load("glove-wiki-gigaword-50")
print(f"Loaded {len(glove.key_to_index):,} word vectors.")

Loading GloVe (first run downloads ~66 MB, under a minute)...
Loaded 400,000 word vectors.


In [35]:
### Insert your word selection from above here
trip_nouns = ['hotel', 'room', 'mattress']
trip_verbs = ['stay', 'recommend', 'apologize']
trip_adj = ['great', 'clean', 'filthy']


for words_to_check in [trip_nouns, trip_verbs, trip_adj]:
  for w in words_to_check:
      if w in glove.key_to_index:
          print(f"\nNearest neighbours for '{w}' in pretrained GloVe:")
          for neighbor, sim in glove.most_similar(w, topn=5):
              print(f"  {neighbor:>12s}   (cosine similarity = {sim:.3f})")
      else:
          print(f"\n'{w}' not in vocabulary.")


Nearest neighbours for 'hotel' in pretrained GloVe:
        hotels   (cosine similarity = 0.835)
    restaurant   (cosine similarity = 0.801)
           inn   (cosine similarity = 0.793)
     apartment   (cosine similarity = 0.781)
         motel   (cosine similarity = 0.779)

Nearest neighbours for 'room' in pretrained GloVe:
       sitting   (cosine similarity = 0.863)
         floor   (cosine similarity = 0.846)
          door   (cosine similarity = 0.846)
         rooms   (cosine similarity = 0.843)
        filled   (cosine similarity = 0.832)

Nearest neighbours for 'mattress' in pretrained GloVe:
    mattresses   (cosine similarity = 0.783)
      cushions   (cosine similarity = 0.758)
        pillow   (cosine similarity = 0.747)
      cupboard   (cosine similarity = 0.744)
       pillows   (cosine similarity = 0.736)

Nearest neighbours for 'stay' in pretrained GloVe:
         leave   (cosine similarity = 0.935)
          wait   (cosine similarity = 0.902)
       staying   (cosi

### 3.1 📝❓ Compare the GloVe neighbours with the ones you retrieved with your CBOW2 (and optionally CBOW5) models above. Do the GloVe neighbours seem more general or semantically broader? Are the CBOW neighbours more domain-specific to hotel reviews or the Sci-Fi text? Discuss how training data size and training objective (predictive vs. count-based) might explain the observed differences.

---

## Lab report

📝❓ Write your lab report here, addressing all questions in the notebook. For convenience, the questions are collected below; please keep your answers clearly marked.

### Questions

**1.3 Preprocessing**

- 📝❓ Describe your decisions for preprocessing the datasets and for the structure of your classes.

**1.5 CBOW5**

- 📝❓ Are predictions made by the model sensitive towards the context size?

**2.1 Hotel reviews**

- 📝❓ List the 5 closest words for each of your 9 chosen words and comment on the performance of your model: do the neighbours the model provides make sense? Discuss.

**2.2 Sci-Fi stories**

- 📝❓ List your findings for the Sci-Fi dataset as well, similarly to 2.1.

**2.3 Comparing the two datasets**

- 📝❓ How does the quality of the hotel review-based embeddings compare with the Sci-fi-based embeddings? Elaborate.

**2.4 Same words, different neighbours**

- 📝❓ Do your 2 chosen words have different neighbours in the hotel review-based and the Sci-fi-based embeddings? If yes, can you reason why?

**2.5 CBOW2 vs. CBOW5**

- 📝❓ What are the differences between CBOW2 and CBOW5? Can you "describe" them?

**3.1 Comparison with GloVe**

- 📝❓ Do the GloVe neighbours seem more general or semantically broader? Are the CBOW neighbours more domain-specific to hotel reviews or the Sci-Fi text? Discuss how training data size and training objective (predictive vs. count-based) might explain the observed differences.

## Declarations

Please fill in both declarations below by editing this cell (double-click it). Mark the applicable option by replacing ☐ with ☒ (copy the symbol from here: ☒) and complete the text field where required. Your reviewers will check that both declarations are present and completed.

### Contribution declaration

By submitting this work, the group declares that the contributions of the group members were:

☐ approximately equal

☐ not approximately equal

If the contributions were not approximately equal, no individual group member needs to be identified.

### Declaration on the use of AI

The group declares that any use of AI-based tools in preparing this work complied with the rules specified for this exercise.

☐ No AI-based tools were used.

☐ AI-based tools were used in accordance with the rules of the exercise.

If AI-based tools were used, briefly state the tool(s) used and the purpose(s) for which they were used:
